initialisation

In [25]:
import time
import importlib
import cv2
import numpy as np
from ugot import ugot
import pose_yolo
from pose_yolo import run_pose_control_inline

# Utilities for showing live camera frames directly inside the notebook
importlib.reload(pose_yolo)
from pose_yolo import run_pose_control_inline

# Utilities for showing live camera frames directly inside the notebook
from IPython.display import display, clear_output, Image
from PIL import Image as Image2

got = ugot.UGOT()
got.initialize("192.168.88.1")

got.load_models(
    [
        "color_recognition",  # detects dominant colors 
        "word_recognition",  # OCR: reads printed text
        "line_recognition",  # for line-following tasks
        "face_recognition",  # identifies registered faces by name
        "apriltag_qrcode" # AprilTag recognition
    ]
)

# Select the default line-tracking mode.
# 0 = single-line mode (follows one line at a time)
got.set_track_recognition_line(0)

# Open the camera stream so later functions can read live frames
got.open_camera()

print("INITIALISATION COMPLETED!")


192.168.88.1:50051
INITIALISATION COMPLETED!


helper functions

ap_centralisation

| Parameter | Default | Description |
|---|---|---|
| `distance` | 0.15 m | Stop when the tag is within this many meters |
| `gap` | 20 px | Pixel tolerance around center before triggering a strafe correction |
| `fwd_spd` | 10 | Forward drive speed (cm/s) |
| `strafe_spd` | 10 | Left/right correction speed (cm/s) |

pick_up
| Joint | Role | Calculation |
|---|---|---|
| `joint1` | Base rotation (left/right) | (AP_x - 320) × -1 / 10 — negative factor corrects for horizontal camera mirroring |
| `joint3` | Extension (reach distance) | AP_distance × 100 - 80 — converts meters to degrees; -80 accounts for resting angle |

In [26]:
def AP_centralization_approaching(distance=0.15, gap=20, fwd_spd=10, strafe_spd=10):
    AP_info = got.get_apriltag_total_info()
    AP_x = AP_info[0][1]
    AP_distance = AP_info[0][6]

    while True:
        AP_info = got.get_apriltag_total_info()

        if not AP_info:  # ← Guard: tag temporarily lost, skip this frame
            got.mecanum_stop()  # Pause to avoid driving blind
            continue            # Try again next iteration

        AP_x = AP_info[0][1]
        AP_distance = AP_info[0][6]

        if AP_x < 320 - gap:
            got.mecanum_move_xyz(-strafe_spd, strafe_spd, 0)
        elif AP_x > 320 + gap:
            got.mecanum_move_xyz(strafe_spd, strafe_spd, 0)
        elif AP_distance > distance:
            got.mecanum_move_xyz(0, fwd_spd, 0)
        else:
            got.mecanum_stop()
            print("It's too close, let's stop.")
            break


def pick_up():
    """
    Pick up the object identified by the closest visible AprilTag.

    Assumes the robot is already aligned and close to the target
    (i.e., AP_centralization_approaching() has just completed).
    """
    # Read the tag's current position and distance for arm targeting.
    AP_info = got.get_apriltag_total_info()
    try:
        AP_x = AP_info[0][1]
        AP_distance = AP_info[0][6]

        # Move arm to a neutral ready position and open the gripper.
        # joint_control(j1, j2, j3, duration_ms): j2=30, j3=30 tilts arm slightly forward.
        got.mechanical_joint_control(0, 30, 30, 1000)
        got.mechanical_clamp_release() # Open gripper before extending arm
        time.sleep(2) # Wait for gripper to fully open

        # Calculate arm joint angles based on the tag's camera position.
        # joint1 (base): convert pixel offset from center to degrees.
        #   Negative factor corrects for the camera being mirrored horizontally.
        joint1 = int((AP_x - 320) * -1 / 10)

        # joint3 (furthest): convert distance (m) to an extension angle.
        # The -80 offset accounts for the arm's resting angle calibration.
        joint3 = int(AP_distance * 100 - 80)

        # Move arm to the computed pick-up pose.
        got.mechanical_joint_control(joint1, 0, joint3, 500)
        print(f"Joint1 value is: {joint1}, Joint3 value is: {joint3}.")
        time.sleep(1) # Wait for arm to reach the target pose

        # Grasp the object and lift the arm back to the carry position.
        got.mechanical_clamp_close()
        time.sleep(2)  # Wait for gripper to fully close before lifting
        got.mechanical_joint_control(0, 30, 30, 1000)  # Return arm to neutral carry pose
    except IndexError:
        print("AprilTag is too close!")

face_and_find_approach

| Parameter | Default | Description |
|---|---|---|
| `gap` | 10 | Pixel tolerance around the center (x = 320 ± gap). Within this band the robot won't strafe. |
| `target_name` | "Stranger" | The name to search for; must match either the face recognition label or text the OCR can read. |
| `turn_spd` | 15 | Speed used when spinning to search for the target. |
| `strafe_spd` | 10 | Sideways speed used to re-center the target in frame during approach. |
| `fwd_spd` | 10 | Forward speed used during approach. |
| `height` | 80 | How tall (in pixels) the face bounding box must be before the robot considers itself close enough to stop. Decrease this to stop further away; increase it to get closer. |
| `adjust_turn` | 10 | How far the robot turns to center itself after first spotting the target, in degree-units. Increase if the robot tends to approach at an angle. |

In [27]:
def face_find_and_approach(
    gap=10, target_name="Stranger", turn_spd=15, strafe_spd=10, fwd_spd=10, height=80, adjust_turn=10
):
    """
    Spin until the target person is found, then approach them.

    Phase 1 — Search:
        The robot turns continuously, checking each frame for the target's
        face (via face_recognition) or name tag (via OCR). When found, it
        stops and does a small corrective turn to face them, then moves to Phase 2.

    Phase 2 — Approach:
        The robot drives toward the face, strafing left/right to keep it
        centered in frame, until the face appears large enough (close enough).
    """
    face_name = None  # Will hold the name from face recognition once a face is found

    try:
        # Phase 1: Spin and search
        while True:
            # Turn slowly to scan the environment
            got.mecanum_turn_speed(turn=3, speed=turn_spd)

            # Read text visible in the frame (e.g. a name tag)
            name = got.get_words_result()

            # Check for any recognized faces in the frame
            faces = got.get_face_recognition_total_info()
            if faces:
                face_name = faces[0][0]  # faces[0] = first face; [0] = its name

            # If either the OCR text or the face name matches our target, we found them!
            if name == target_name or face_name == target_name:
                got.mecanum_stop()
                print(f"Saw {target_name}!")

                # Small corrective turn to center the robot on the target
                # turn=3 is clockwise; times=10, unit=2 means ~10 degree-units
                got.mecanum_turn_speed_times(turn=3, speed=20, times=adjust_turn, unit=2)
                break  # Exit Phase 1, move on to Phase 2

        # Phase 2: Approach the target
        while True:
            name = got.get_words_result()
            faces = got.get_face_recognition_total_info()

            if not faces:
                # Lost the face; inch forward slowly to try to find it again
                got.mecanum_translate_speed(angle=0, speed=fwd_spd)
            else:
                c_x = faces[0][1]  # Horizontal center of the face in the frame (0–640 px)
                h = faces[0][3]  # Height of the face bounding box (proxy for distance)
                if h < height:
                    if c_x < 320 - gap:
                        # Face is too far LEFT — strafe left while moving forward
                        got.mecanum_move_xyz(
                            x_speed=-strafe_spd, y_speed=fwd_spd, z_speed=0
                        )
                    elif c_x > 320 + gap:
                        # Face is too far RIGHT — strafe right while moving forward
                        got.mecanum_move_xyz(x_speed=strafe_spd, y_speed=fwd_spd, z_speed=0)
                    else:
                        # Face is centered but still small (far away) — move straight forward
                        got.mecanum_move_xyz(x_speed=0, y_speed=fwd_spd, z_speed=0)
                else:
                    # Face is centered AND large enough — we've arrived!
                    got.mecanum_stop()
                    print(f"Reached {name}!")
                    break  # Done

            clear_output(wait=True)

        got.mecanum_stop()

    except KeyboardInterrupt:
        print("Done")
        got.mecanum_stop()

line_follow

| Parameter | Default | Description |
|---|---|---|
| `mult` | 0.25 | Multiplier converting line offset into rotation speed. Higher = sharper corrections; lower = gentler steering. |
| `speed` | 35 | Forward speed while following the line. |

**Return values:** `(line_type, x, y)`
| Value | Description |
|---|---|
| `line_type` | `0` = no line detected, `1` = normal line, `2` = y-intersection (fork in the path) |
| `x`, `y` | Pixel coordinates of the detected line position |

In [28]:
def line_follow(mult=0.25, speed=35):
    """Follow the detected line by turning proportionally to the line offset."""
    # Read line-tracking information from the robot.
    # `offset` tells how far the detected line is from the center.
    # `type` describes the line/intersection pattern detected.
    # `x` and `y` are the detected line position coordinates.
    offset, line_type, x, y = got.get_single_track_total_info()

    # Convert the line offset into a turning speed.
    # Larger offset -> stronger rotation to re-center on the line.
    rotation_speed = int(offset * mult)

    # Move forward while rotating to stay aligned with the line.
    got.mecanum_move_xyz(x_speed=0, y_speed=speed, z_speed=rotation_speed)

    # Return the detection info so the main program can make decisions.
    return line_type, x, y

In [29]:
got.set_track_recognition_line(0)
got.open_camera()

try:
    while True:
        line_type, _, _ = line_follow(mult=0.25, speed=20)

        if line_type == 0:  # 0 = no line detected,
            print("No line!")
            break

        if line_type == 2:  # 2 = y-intersection detected
            print("Y-intersection!")
            break

    got.mecanum_stop()

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done!")

No line!


combined code

In [30]:
red_seen = False  # Flag to track if Red has been detected yet
got.set_track_recognition_line(line_type=0)  # 0 is a single track line

try:
    while True:
        color = got.get_color_total_info()[0]  # Read fresh color data each loop

        if not red_seen:
            if color == "Red":
                got.screen_display_background(3)  # Red background
                print("Red detected!")
                red_seen = True  # Unlock green detection
        else:
            if color == "Green":
                got.screen_display_background(6)  # Green background
                print("Green detected!")
                time.sleep(1)
                break  # Both colors seen — exit loop

    while True:
        AP_info = got.get_apriltag_total_info()

        if AP_info:
            AP_centralization_approaching(
                distance=0.14, gap=18, fwd_spd=11, strafe_spd=5
            )
            pick_up()
            print("Picked up!")
            time.sleep(2)
            break
        else:
            got.mecanum_move_speed(0, 15)

    # Line follow pt 1
    print("Starting line following...")
    while True:
        got.mecanum_move_speed_times(direction=0, speed=20, times=15, unit=1)
        line_type, _, _ = line_follow(mult=0.25, speed=20)

        if line_type == 0:  # 0 = no line detected,
            print("No line!")
            break

        if line_type == 2:  # 2 = y-intersection detected
            print("Y-intersection!")
            break

    got.mecanum_stop()

    # If you want to go forward a bit to move closer to the text:
    got.mecanum_move_speed_times(direction=0, speed=20, times=20, unit=1)

    # or turn
    got.mecanum_turn_speed_times(turn=2, speed=30, times=15, unit=2)

    # Text Recognition
    print("Starting text recognition...")
    while True:
        text = got.get_words_result()

        print("Text:", text)

        # React to specific command words
        if text == "LEFT":
            # Turn counter-clockwise by ~45 degrees
            got.mecanum_turn_speed_times(turn=2, speed=20, times=45, unit=2)
            break
        elif text == "RIGHT":
            # Turn clockwise by ~45 degrees
            got.mecanum_turn_speed_times(turn=3, speed=20, times=45, unit=2)
            break

    # Line follow pt 2
    print("Starting line follow...")
    while True:
        line_type, _, _ = line_follow(mult=0.25, speed=15)

        if line_type in [0]:
            break

    # Pose recognition
    print("Starting pose recognition...")
    run_pose_control_inline(
        robot_ip="192.168.1.124",
        forward_speed=30,
        backward_speed=30,
        turn_speed=45,
        camera_index=0,
        model_path="yolov8n-pose.pt",
        up_margin_factor=0.6,
        down_margin_factor=0.6,
        min_conf=0.3,
        enable_robot=True,  # <-- Robot is ENABLED: it will now move!
        debounce_frames=2,
        max_frames=None,
        got=got,
    )
    time.sleep(2)
    clear_output()

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done!")
    